# Suite DOM — Value domains

A literal carries its value domain: a standard's representation (`Integer`, `Ieee754`, `Bits`, `Bytes`, `Unicode`,
`Ieee1164`, `Enum`, `Packed`, `Bool`), or by default its native's. Domains are data, compared by structure and
validated; a registered domain is written by name, any other by value, and a default one not at all. Evaluating
literals of other domains comes in later phases.

In [ ]:
from mbse.Expressions import Domains as D, Evaluators as V, Expressions as E
from mbse.Expressions import Translators as T
from mbse.Expressions.Dialects.Python import Expressions as P
from mbse.Expressions.Framework import Domains as FD, Terms as F
from mbse.Schemas.Framework import JSON, YAML, Comparison as C, Plain, Validators
from support import raises

I8 = D.OfInteger.Builder().width(8).create()
U8 = D.OfInteger.Builder().width(8).signed(False).overflow("wrap").create()
F32 = D.OfIeee754.Builder().format("binary32").create()
Bits2 = D.OfBits.Builder().width(2).create()
State = D.OfEnum.Builder().members("IDLE", "RUN").create()
Coded = D.OfPacked.Builder().domain(State).representation(Bits2).codes(0, 1).create()


def snapshot(literal):
    """A literal's snapshot, as plain data."""
    return Plain.ToPlain(E.OfLiteral.Schema, literal)["objects"]["s0"]

## DOM-01 · Value domains name standards, with only their parameters, are compared by structure and validated

In [ ]:
names = [(D.Bool, "bool"), (D.Int, "int"), (I8, "int8"), (U8, "uint8"), (D.OfInteger.Builder().signed(False).create(), "uint"),
         (D.Float, "float"), (F32, "binary32"), (D.OfIeee754.Builder().rounding("roundTowardZero").create(),
                                                 "binary64 roundTowardZero"),
         (Bits2, "bits2"), (D.Bytes, "bytes"), (D.OfBytes.Builder().width(4).create(), "bytes4"), (D.Str, "str"),
         (D.OfIeee1164.Builder().create(), "std_logic"), (State, "enum(IDLE, RUN)"), (Coded, "packed(enum(IDLE, RUN) as bits2)")]
assert [(domain.name(), repr(domain)) for domain, _ in names] == [(name, name) for _, name in names]
assert all(domain.validate() == [] for domain, _ in names)
# The natives' defaults, compared by structure: an equal domain built anew is the same domain.
assert D.OfInteger.Data() == D.Int and D.OfIeee754.Data("binary64") == D.Float and D.OfUnicode.Data() == D.Str
assert D.OfBytes.Data() == D.Bytes and D.OfBool.Data() == D.Bool and I8 != U8 and I8 != D.Int
assert D.OfInteger.Builder().width(8).create() == I8 and repr(D.OfInteger) == "<domain kind integer>"
assert all(kind.Schema.validate() == [] for kind in [D.OfBool, D.OfInteger, D.OfIeee754, D.OfBits, D.OfBytes, D.OfUnicode,
                                                     D.OfIeee1164, D.OfEnum, D.OfPacked]) and D.Schema.validate() == []
assert [b.name for b in D.Schema.branches] == ["bool", "integer", "ieee754", "bits", "bytes", "unicode", "ieee1164", "enum",
                                               "packed", "named"]
assert D.FORMATS[2] == "binary64" and D.ROUNDINGS[0] == "roundTiesToEven" and D.OVERFLOWS == ("wrap", "saturate", "raise")
assert len(D.STATES) == 9
# Each kind's problems.
problems = [
    (D.OfInteger.Data(0), ["a width must be a positive int, got 0"]),
    (D.OfInteger.Data(8, "yes", "clip"), ["signed must be a bool, got 'yes'", "overflow must be one of wrap, saturate, raise, got 'clip'"]),
    (D.OfIeee754.Data("binary8", "up"), [
        "format must be one of binary16, binary32, binary64, binary128, decimal64, decimal128, got 'binary8'",
        "rounding must be one of roundTiesToEven, roundTiesToAway, roundTowardPositive, roundTowardNegative, "
        "roundTowardZero, got 'up'"]),
    (D.OfBits.Data(), ["a bits domain needs a width"]),
    (D.OfBits.Data(0), ["a width must be a positive int, got 0"]),
    (D.OfBytes.Data(-1), ["a width must be a positive int, got -1"]),
    (D.OfEnum.Data(), ["an enum needs a member"]),
    (D.OfEnum.Data(("A", "", "A")), ["member 1 has no name", "member 'A' appears twice"]),
    (D.OfPacked.Data(I8, Bits2, ()), ["a packed domain packs an enum, got int8"]),
    (D.OfPacked.Data(), ["a packed domain packs an enum, got None"]),
    (D.OfPacked.Data(State, D.Bytes, (0, 1)), ["a packed domain's representation is an integer or bits of a width, got bytes"]),
    (D.OfPacked.Data(State, D.Int, (0, 1)), ["a packed domain's representation is an integer or bits of a width, got int"]),
    (D.OfPacked.Data(State, Bits2, (0,)), ["a packed domain needs a code per member: 2 members, 1 codes"]),
    (D.OfPacked.Data(State, D.OfBits.Data(1), (0, 2)), ["code 1 does not fit bits1: 2"]),
    (D.OfPacked.Data(State, Bits2, (1, 1)), ["code 1 is used twice"]),
    (D.OfPacked.Data(State, D.OfInteger.Data(2), (-2, 2)), ["code 1 does not fit int2: 2"]),
    (D.OfPacked.Data(State, D.OfInteger.Data(2), ("0", 1)), ["code 0 does not fit int2: '0'"]),
    (D.OfPacked.Data(D.OfEnum.Data(), D.OfInteger.Data(2, True, "x"), ()), [
        "domain: an enum needs a member", "representation: overflow must be one of wrap, saturate, raise, got 'x'"]),
]
for domain, expected in problems:
    assert domain.validate() == expected, (domain, domain.validate())
assert D.OfPacked.Data().name() == "packed(None as None)"
# Builders finalize as every builder does, and set each parameter.
built = D.OfIeee754.Builder(F32).rounding("roundTowardNegative").clone()
assert built == D.OfIeee754.Data("binary32", "roundTowardNegative") and F32.rounding == "roundTiesToEven"
assert D.OfBytes.Builder(D.OfBytes.Data()).width(3).update().width == 3
for builder in [D.OfBool.Builder(), D.OfUnicode.Builder(), D.OfIeee1164.Builder()]:
    assert builder.create().validate() == []
with raises(ValueError, match="create() is only valid without a source instance; use clone() or update()"):
    D.OfInteger.Builder(I8).create()
with raises(ValueError, match="clone() is only valid with a source instance"):
    D.OfInteger.Builder().clone()
with raises(ValueError, match="update() is only valid with a source instance"):
    D.OfInteger.Builder().update()
with raises(TypeError, match="expected integer data to build from, got bits2"):
    D.OfInteger.Builder(Bits2)

## DOM-02 · A domain holds the values of its native type that fit it, and includes only itself

In [ ]:
holds = [
    (I8, [127, -128], [128, -129, True, 1.0]), (U8, [0, 255], [-1, 256]), (D.Int, [10 ** 30, -5], [False]),
    (D.OfInteger.Data(signed=False), [10 ** 30], [-1]), (D.OfInteger.Data(0), [10 ** 30], []),  # not a width: unbounded
    (F32, [1.5, float("nan")], [1, "1.5"]), (D.OfIeee754.Data("decimal64"), ["0.1"], [0.1]),
    (D.OfBits.Data(3), [b"\x07", b"\x00"], [b"\x08", b"\x00\x07", "7"]), (D.OfBits.Data(8), [b"\xff"], [b""]),
    (D.OfBits.Data(9), [b"\x01\xff"], [b"\x02\x00", b"\xff"]), (D.OfBits.Data(), [], [b"\x00"]),
    (D.OfBytes.Data(2), [b"ab"], [b"a"]), (D.Bytes, [b"", b"abc"], ["abc"]),
    (D.Str, ["", "日本語 🚀"], ["\ud800", b"a"]), (D.OfIeee1164.Data(), ["Z", "-", "U"], ["z", "2", 1]),
    (State, ["RUN"], ["STOP", 0]), (Coded, ["IDLE"], ["STOP"]), (D.OfPacked.Data(), [], ["IDLE"]),
    (D.Bool, [True, False], [1, 0]),
]
for domain, inside, outside in holds:
    assert all(domain.contains(v) for v in inside) and not any(domain.contains(v) for v in outside), domain
assert D.OfIeee754.Data("decimal128").native() is str and F32.native() is float and I8.native() is int
# A domain includes only domains equal to it: Basic has no implied promotions.
assert I8.includes(D.OfInteger.Data(8)) and not I8.includes(D.Int) and not D.Int.includes(I8) and not I8.includes(U8)
assert I8.includes(FD.OfUnion(I8, D.OfInteger.Data(8))) and not I8.includes(FD.OfUnion(I8, U8))
assert FD.overlaps(FD.Anything, I8) and not D.Float.includes(F32)
# A native's default domain, and the domains of other values.
assert [D.of(v) for v in [True, 1, 1.5, "a", b"a"]] == [D.Bool, D.Int, D.Float, D.Str, D.Bytes]
assert D.of(E.variable("x").data) is D.Object
with raises(TypeError, match="a NoneType is not a value of the Basic dialect"):
    D.of(None)

## DOM-03 · Domains are registered by name, once, and written by name; any other by value

In [ ]:
D.register("uint8", U8)
assert D.registered("uint8") is U8 and D.name_of(U8) == "uint8" and D.name_of(D.OfInteger.Data(8, False, "wrap")) == "uint8"
assert D.name_of(I8) is None
with raises(ValueError, match="domain 'uint8' is already registered"):
    D.register("uint8", I8)
with raises(ValueError, match="uint8 is already registered as 'uint8'"):
    D.register("byte", D.OfInteger.Data(8, False, "wrap"))
D.register("uint16", D.OfInteger.Data(16, False))  # another domain: each name, and each domain, once
assert D.name_of(D.OfInteger.Data(16, False)) == "uint16"
with raises(LookupError, match="no domain registered as 'nope'"):
    D.registered("nope")
assert D.to_plain(U8) == {"named": {"name": "uint8"}}
assert D.to_plain(I8) == {"integer": {"width": 8, "signed": True, "overflow": "raise"}}
assert D.to_plain(D.Int) == {"integer": {"signed": True, "overflow": "raise"}}
assert D.to_plain(D.OfPacked.Data(State, U8, (0, 1))) == {
    "packed": {"domain": {"enum": {"members": ["IDLE", "RUN"]}}, "representation": {"named": {"name": "uint8"}}, "codes": [0, 1]}}
assert D.to_plain(D.OfUnicode.Data()) == {"unicode": {}}
for domain in [U8, I8, F32, Bits2, D.Bytes, D.Str, State, Coded, D.OfPacked.Data(State, U8, (0, 1)), D.Bool]:
    assert D.from_plain(D.to_plain(domain)) == domain
with raises(TypeError, match="not a domain: 'int8'"):
    D.to_plain("int8")
with raises(LookupError, match="no domain registered as 'nope'"):
    D.from_plain({"named": {"name": "nope"}})

## DOM-04 · A literal carries its domain; a native's default domain is no domain at all

In [ ]:
lit = E.literal(200, U8).data
assert lit.domain is U8 and lit.typed() is U8 and lit.validate() == []
assert E.literal(5, D.Int).data.domain is None and E.literal(5).data.typed() is None  # the default is left out
assert E.OfLiteral.Builder().value(-1).domain(I8).create().domain == I8
assert E.OfLiteral.Builder(lit).value(7).clone().domain is U8 and E.OfLiteral.Builder(lit).domain(None).clone().domain is None
assert E.OfLiteral.Builder(E.literal(1).data).domain(D.Int).update().domain is None
# A literal's domain is valid and holds its value.
assert E.literal(300, U8).data.validate() == ["a literal of uint8 cannot hold 300"]
assert E.literal(b"\x00", D.OfBits.Data()).data.validate() == ["domain: a bits domain needs a width"]
assert E.OfLiteral.Data(1, "int8").validate() == ["a literal's domain must be a value domain, got str"]
assert E.operation("eq", E.literal("RUN", Coded), E.literal("STOP", Coded)).data.validate() == [
    "argument 1: a literal of packed(enum(IDLE, RUN) as bits2) cannot hold 'STOP'"]
# Inference gives a literal's domain, and Basic does not promote: int8 and int8 add only once arithmetic takes them.
assert E.DIALECT.infer(lit) is U8 and E.DIALECT.infer(E.literal(1)) is D.Int
with raises(TypeError, match="add cannot take (int8, int8); it takes (T, T) -> T for T in int | float"):
    E.DIALECT.infer(E.operation("add", E.literal(1, I8), E.literal(2, I8)))
# Forms, `make` and `same` carry the domain.
assert lit.form().attributes == {"value": 200, "domain": U8}
assert F.same(E.DIALECT.make(lit.form()), lit) and F.same(lit, E.literal(200, D.OfInteger.Data(8, False, "wrap")).data)
assert not F.same(lit, E.literal(200, I8).data) and not F.same(lit, E.literal(200).data)
# Evaluation of other domains comes later; a default domain evaluates as before.
with raises(NotImplementedError, match="literals of uint8 are not evaluated yet"):
    V.OfAny(lit)
assert V.OfAny(E.operation("add", E.literal(1, D.Int), 2)) == 3
# Translation keeps a domain only where the other dialect has one: none does yet.
with raises(ValueError, match="Basic literal 200 of uint8 has no Python counterpart"):
    T.between(E.DIALECT, P.DIALECT).forward(lit)

## DOM-05 · Over the wire a literal's domain is written by name when registered, otherwise by value

In [ ]:
assert snapshot(E.literal(200, U8).data) == {"kind": "literal", "int": 200, "domain": {"named": {"name": "uint8"}}}
assert snapshot(E.literal(-1, I8).data) == {
    "kind": "literal", "int": -1, "domain": {"integer": {"width": 8, "signed": True, "overflow": "raise"}}}
assert snapshot(E.literal(5, D.Int).data) == {"kind": "literal", "int": 5}
text = JSON.ToJSON(E.OfLiteral.Schema, E.literal("RUN", Coded).data)
assert text == ('{"root": "s0", "objects": {"s0": {"kind": "literal", "str": "RUN", "domain": {"packed": {"domain": '
                '{"enum": {"members": ["IDLE", "RUN"]}}, "representation": {"bits": {"width": 2}}, "codes": [0, 1]}}}}}')
for literal in [E.literal(200, U8).data, E.literal(-1, I8).data, E.literal("RUN", Coded).data, E.literal(1.5, F32).data,
                E.literal("Z", D.OfIeee1164.Data()).data, E.literal(b"\x03", Bits2).data]:
    back = JSON.FromJSON(E.Builders)(E.OfLiteral.Schema, JSON.ToJSON(E.OfLiteral.Schema, literal))
    assert F.same(back, literal) and back.domain == literal.domain
    assert F.same(YAML.FromYAML(E.Builders)(E.OfLiteral.Schema, YAML.ToYAML(E.OfLiteral.Schema, literal)), literal)
    assert Validators.Validate(E.Builders)(E.OfLiteral.Schema, back) == []
assert JSON.FromJSON(E.Builders)(E.OfLiteral.Schema, JSON.ToJSON(E.OfLiteral.Schema, E.literal(1, U8).data)).domain is U8
with raises(LookupError, match="no domain registered as 'nope'"):
    Plain.FromPlain(E.Builders)(E.OfLiteral.Schema, {"root": "s0", "objects": {"s0": {
        "kind": "literal", "int": 1, "domain": {"named": {"name": "nope"}}}}})
# Literals compare under their meta-schema with their domains.
compare = lambda a, b: C.OfObject(E.OfLiteral.Schema, a).compare(C.OfObject(E.OfLiteral.Schema, b))  # noqa: E731
assert compare(E.literal(1, I8).data, E.literal(1, D.OfInteger.Data(8)).data) == 0
assert compare(E.literal(1, I8).data, E.literal(1, U8).data) is None and compare(E.literal(1, I8).data, E.literal(1).data) is None
# The builder reads and writes the domain through the visitor protocols too.
def domain_field(builder, branch, name, act):
    """Calls `act` with the native handle of the domain's field `name`, in the union branch `branch`."""
    builder.property("domain", lambda p: p.value(lambda a: a.as_union(lambda u: u.property(branch, lambda q: q.value(
        lambda b: b.as_object(lambda o: o.property(name, lambda w: w.value(lambda n: n.as_native(act)))))))))


builder = E.OfLiteral.Builder(E.literal(-1, I8).data)
names = []
builder.properties(lambda p: names.append(p.name()))
assert builder.has("domain") and names == ["kind", "int", "domain"]
widths = []
domain_field(builder, "integer", "width", lambda n: widths.append(n.get()))
assert widths == [8]
domain_field(builder, "integer", "width", lambda n: n.set(16))
assert builder.clone().domain == D.OfInteger.Data(16)
builder.property("domain", lambda p: p.clear())
assert not builder.has("domain") and builder.clone().domain is None
domain_field(builder, "named", "name", lambda n: n.set("uint8"))
assert builder.clone().domain is U8
builder.clear("domain")
assert builder.clone().domain is None
builder.property("domain", lambda p: p.value(lambda a: a.as_union(lambda u: None)))  # a union without a branch is no value
assert builder.clone().domain is None